# V6: F1 + v20 확신도 (어려운 한글 행 → v20, 단 v20의 1·2위 확률 차이 < TAU면 112)
**GPU 런타임(L4)**으로 위에서부터 실행하세요. 끝나면 `out/`에 `V6_t10.zip`(TAU 0.1), `V6_t30.zip`(TAU 0.3)이 생깁니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
D = "/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/combo_kit"
os.chdir(D)
!ls -la build_fuse_conf.py build_fuse.py v20.zip 112_submit.zip check_combo.py
!nvidia-smi --query-gpu=name --format=csv
!pip -q install lightgbm==4.6.0 catboost==1.2.8 rapidfuzz==3.14.3 transformers==4.57.6 accelerate==1.9.0 scikit-learn==1.5.2


In [ ]:
T_c = 18.2   # 115(C_vote) 서버 실행 시간(분)
BUD = int(min(24, 28.5 - max(0.0, T_c - 11.8)))
BUD = max(BUD, 18)
print("v20 시간 예산(분):", BUD)
W = "/content/combo"; os.makedirs(W, exist_ok=True); os.makedirs(f"{D}/out", exist_ok=True)


In [ ]:
# 조립 (약 15분) → TAU만 바꾼 사본 (약 4분)
for f in ("V6_t10.zip", "V6_t30.zip"):
    if os.path.exists(f"{W}/{f}"): os.remove(f"{W}/{f}")
!python build_fuse_conf.py {W}/V6_t10.zip v20.zip 112_submit.zip 4 {BUD} 0.1
!python build_fuse_conf.py --tau {W}/V6_t10.zip {W}/V6_t30.zip 0.3
!unzip -p {W}/V6_t10.zip model/ko/script.py | grep -n "v20_margin\|conf\[i\]"


In [ ]:
# 점검 (약 10분): 기준 ZIP은 3,000행 + 영문 불변 + 600행 행 독립성 + v20 확신도 출력 확인. 사본은 TAU 한 줄 말고 같은지 확인
import zipfile, difflib
zp = f"{W}/V6_t10.zip"
c = subprocess.run([sys.executable, "check_combo.py", zp, f"{W}/work_v6"], capture_output=True, text=True)
log = c.stdout + "\n" + c.stderr
open(f"{W}/V6_check.log", "w").write(log)
print("\n".join(l for l in log.splitlines() if l.startswith(("[1]", "[2]", "[3]", "zip size", "router start", "OK", "PROBLEM", "WARNING")) or "predictions to output" in l))
bad = [k for k in ("WARNING ko", "WARNING en", "still running", "did not start", "en failed") if k in log]
last = (c.stdout.strip().splitlines() or [""])[-1]
ok = c.returncode == 0 and last == "OK" and not bad and log.count("fell back 0,") == 2 and "unsure->en" in log and os.path.getsize(zp) < 10e9
def same_but_tau(a, b):
    za, zb = zipfile.ZipFile(a), zipfile.ZipFile(b)
    ia = {i.filename: (i.file_size, i.CRC) for i in za.infolist() if i.filename != "script.py"}
    ib = {i.filename: (i.file_size, i.CRC) for i in zb.infolist() if i.filename != "script.py"}
    d = [l for l in difflib.unified_diff(za.read("script.py").decode().splitlines(), zb.read("script.py").decode().splitlines(), lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")]
    return ia == ib and len(d) == 2 and all(l[1:].startswith("TAU = ") for l in d)
print("기준 ZIP:", "PASS" if ok else f"FAIL {bad or last}")
if ok:
    for name in ("V6_t10.zip", "V6_t30.zip"):
        if name == "V6_t10.zip" or same_but_tau(zp, f"{W}/{name}"):
            shutil.copy(f"{W}/{name}", f"{D}/out/{name}"); print(name, ": PASS ->", round(os.path.getsize(f"{D}/out/{name}")/1e9, 2), "GB")
        else:
            print(name, ": FAIL - 올리지 마세요")
else:
    print("V6: FAIL - 올리지 마세요. 전체 로그:", f"{W}/V6_check.log")
